# Find Clumps — Identifying Frequent k-mers in Genomic Windows

**Topic:** DNA Sequence Analysis and Clump Finding

## Overview

This notebook implements two approaches to identify k-mers that occur frequently within a local region of a DNA sequence.

The problem is known as the **Clump Finding Problem**.

A k-mer forms an `(L, t)-clump` if it appears at least `t` times within at least one window of length `L`.

Two implementations are presented:

1. `FindClumps()` — a direct approach that recalculates k-mer frequencies for each window.
2. `FindClumpsOptimized()` — an optimized approach that updates k-mer frequencies as the window moves along the sequence.

Both implementations are preserved to demonstrate algorithmic development and optimization.

## Objectives

- Understand the Clump Finding Problem.
- Identify frequently occurring k-mers within genomic windows.
- Apply modular programming by reusing frequency-counting functions.
- Implement and compare direct and sliding-window algorithms.
- Understand how Python dictionaries and sets support sequence analysis.
- Validate both implementations using automated tests.

## Biological Background and Study Notes

### The Clump Finding Problem

The Clump Finding Problem is more complex than simple pattern counting because it considers the local distribution of k-mers.

Instead of counting occurrences across an entire DNA sequence, the algorithm examines overlapping windows of a specified length.

For each window, it identifies k-mers appearing at least `t` times.

A k-mer is included in the final result if it satisfies this condition in at least one window.

### Understanding the Parameters

- `Text`: DNA sequence to analyze.
- `k`: Length of each nucleotide pattern.
- `L`: Length of the sliding window.
- `t`: Minimum number of occurrences required to form a clump.

For example, with `k = 9`, `L = 500`, and `t = 3`, the algorithm searches for 9-mers appearing at least three times within a window of 500 nucleotides.

### Modularity in Programming

Writing a complete clump-finding algorithm from scratch can be challenging.

Modular programming makes this task easier by dividing the problem into smaller, reusable functions.

The `FrequencyMap()` function calculates k-mer frequencies within a sequence.

By applying this function to each window of length `L`, we can identify k-mers whose frequencies meet or exceed `t`.

New qualifying k-mers are added to the collection of identified patterns.

At the end, the result contains all distinct k-mers that form `(L, t)-clumps`.

This approach demonstrates how previously developed functions can be reused to solve more complex computational problems.

## Algorithm 1 — Direct Clump Finding

### Pseudocode

```text
FindClumps(Text, k, L, t)
    Patterns ← an empty collection
    n ← length(Text)

    for every integer i between 0 and n - L
        Window ← Text[i:i + L]
        freqMap ← FrequencyMap(Window, k)

        for every k-mer s in freqMap
            if freqMap[s] ≥ t
                add s to Patterns if not already present

    return Patterns
```

### Step-by-Step Explanation

1. Initialize an empty collection of patterns.
2. Move a window of length `L` along the DNA sequence.
3. Calculate the frequency of every k-mer within the current window.
4. Identify k-mers appearing at least `t` times.
5. Add qualifying k-mers to the result without duplicates.
6. Continue until all complete windows have been examined.
7. Return the collection of clump-forming k-mers.

### Implementation Notes

The direct implementation uses `FrequencyMap()` to calculate the frequency distribution within each window.

A list preserves the order in which qualifying k-mers are first encountered.

A set named `seen` prevents duplicates from being added to the result.

This approach is straightforward and useful for understanding the algorithm, although rebuilding the frequency map for every window introduces repeated calculations.

In [1]:
def FrequencyMap(Text, k):
    """
    Count the frequency of each k-mer in a DNA sequence.
    Overlapping occurrences are included.
    """
    if not isinstance(Text, str):
        raise ValueError("Text must be a string.")
    if not isinstance(k, int) or isinstance(k, bool) or k <= 0:
        raise ValueError("k must be a positive integer.")

    freq = {}

    for i in range(len(Text) - k + 1):
        Pattern = Text[i:i + k]
        freq[Pattern] = freq.get(Pattern, 0) + 1

    return freq


def validate_clump_inputs(Text, k, L, t):
    """Validate the input parameters for clump finding."""
    if not isinstance(Text, str):
        raise ValueError("Text must be a string.")
    for name, value in [("k", k), ("L", L), ("t", t)]:
        if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    if k > L:
        raise ValueError("k cannot exceed the window length L.")


def FindClumps(Text, k, L, t):
    """
    Find k-mers occurring at least t times within an L-length window.
    This implementation recomputes the frequency map for each window.

    Returns
    -------
    list[str]
        Distinct k-mers forming (L, t)-clumps.
    """
    validate_clump_inputs(Text, k, L, t)

    if L > len(Text):
        return []

    patterns = []
    seen = set()

    for i in range(len(Text) - L + 1):
        window = Text[i:i + L]
        freq_map = FrequencyMap(window, k)
        for kmer, count in freq_map.items():
            if count >= t and kmer not in seen:
                patterns.append(kmer)
                seen.add(kmer)
    return patterns


# with open('E_coli.txt', 'r') as file:
#     Text = file.read().strip() 
#     k = 9
#     L = 500
#     t = 3
#     print(FindClumps(Text, k, L, t))

## Algorithm 2 — Optimized Clump Finding

### Sliding-Window Approach

The optimized implementation uses a sliding-window technique to avoid recalculating the entire frequency map whenever the window moves.

First, the algorithm builds the frequency map for the initial window of length `L`.

It then moves the window one nucleotide at a time.

At each step:

1. Identify the k-mer leaving the previous window.
2. Decrease its occurrence count.
3. Remove its dictionary entry if the count reaches zero.
4. Identify the k-mer entering the new window.
5. Increase its occurrence count.
6. Check whether the incoming k-mer reaches the threshold `t`.

Because the frequencies of the remaining k-mers are unchanged, there is no need to rebuild the entire dictionary.

### Using defaultdict

The implementation uses `defaultdict(int)` from Python's `collections` module.

Unlike a regular dictionary, `defaultdict(int)` automatically initializes a missing key with the integer value zero.

For example:

```python
from collections import defaultdict

freq_map = defaultdict(int)

freq_map["ATG"] += 1
```

The first occurrence of `ATG` is automatically counted as one.

### Avoiding Duplicate Results

The optimized implementation stores qualifying k-mers in a set:

```python
patterns = set()
```

A set automatically prevents duplicate entries.

The function returns the sorted results to provide a reproducible output order.

### Comparing the Approaches

| Feature | Direct approach | Optimized approach |
|---|---|---|
| Frequency counting | Recalculated for every window | Updated incrementally |
| Window processing | Rebuilds the frequency map | Removes and adds k-mers |
| Duplicate handling | List and set | Set |
| Main advantage | Easier to understand | Avoids repeated calculations |

The optimized approach is particularly useful for longer genomic sequences where many overlapping windows need to be examined.

In [2]:
from pathlib import Path

# Inspect the Salmonella enterica dataset
dataset_path = Path("../dataset/Salmonella_enterica.txt")

with dataset_path.open("r", encoding="utf-8") as file:
    content = file.read()

print("File size (bytes):", dataset_path.stat().st_size)
print("Number of lines:", len(content.splitlines()))
print("First 150 characters:", repr(content[:150]))
print("Last 150 characters:", repr(content[-150:]))
print("Sequence length after strip():", len(content.strip()))

File size (bytes): 60
Number of lines: 1
First 150 characters: 'CCCGGATCC GGATCCGGG CGGATCCGG CCGGATCCG GGATCCTGG CCAGGATCC '
Last 150 characters: 'CCCGGATCC GGATCCGGG CGGATCCGG CCGGATCCG GGATCCTGG CCAGGATCC '
Sequence length after strip(): 59


In [3]:
from collections import defaultdict

def FrequencyMapSliding(Text, k):
    """
    Build a k-mer frequency map using a defaultdict.
    This helper illustrates frequency counting with automatically
    initialized integer values.
    """
    return defaultdict(int, FrequencyMap(Text, k))


def FindClumpsOptimized(Text, k, L, t):
    """
    Find (L, t)-clumps using a sliding-window frequency map.
    Instead of rebuilding the map for each window, update the
    counts of the outgoing and incoming k-mers.

    Returns
    -------
    list[str]
        Distinct k-mers forming (L, t)-clumps.
    """
    validate_clump_inputs(Text, k, L, t)

    if L > len(Text):
        return []

    patterns = set()
    freq_map = FrequencyMapSliding(Text[:L], k)

    # Check the first window
    for kmer, count in freq_map.items():
        if count >= t:
            patterns.add(kmer)
    # Slide the window one nucleotide at a time
    for i in range(1, len(Text) - L + 1):
        # Remove the outgoing k-mer
        outgoing_kmer = Text[i - 1:i - 1 + k]
        freq_map[outgoing_kmer] -= 1

        if freq_map[outgoing_kmer] == 0:
            del freq_map[outgoing_kmer]
        # Add the incoming k-mer
        incoming_kmer = Text[i + L - k:i + L]
        freq_map[incoming_kmer] += 1

        # Check whether the new k-mer forms a clump
        if freq_map[incoming_kmer] >= t:
            patterns.add(incoming_kmer)
    return sorted(patterns)


## Validation and Testing

Both clump-finding implementations were evaluated using eight automated tests.

### Test Cases

1. **Basic clump finding:** Identify qualifying k-mers in a representative DNA sequence.
2. **Optimized implementation:** Verify the result produced by the sliding-window approach.
3. **Repeated k-mers:** Identify a clump formed by repeated patterns within a window.
4. **No clumps:** Return an empty collection when no k-mer satisfies the threshold.
5. **Implementation agreement:** Compare the direct and optimized approaches.
6. **Invalid inputs:** Reject invalid values and unsupported parameter types.
7. **Short sequences:** Handle sequences shorter than the requested window length.
8. **Edge cases:** Confirm agreement between implementations across multiple parameter combinations.

### Results

All eight tests passed.

Both implementations identified the same sets of qualifying k-mers in the tested cases.

### Computational Considerations

Let `n` represent the sequence length, `L` the window length, and `k` the k-mer length.

The direct approach recalculates the frequency map for each complete window. Its worst-case time complexity is O((n - L + 1)(L - k + 1)k).

The optimized approach constructs the initial frequency map and updates it as the window moves. Its expected time complexity is O((L - k + 1)k + (n - L)k), assuming average constant-time dictionary operations.

This illustrates how sliding-window algorithms can reduce repeated computation when analyzing long sequences.

In [4]:
# Test 1: Basic clump finding
test_text = "CGGACTCGACAGATGTGAAGAAATGTGAAGACTGAGTACGCGCGCG"
expected = {"CGCG"}
result = set(FindClumps(test_text, 4, 25, 3))
assert result == expected
print("Test 1 passed: Basic clump finding")

# Test 2: Optimized implementation
result = set(FindClumpsOptimized(test_text, 4, 25, 3))
assert result == expected
print("Test 2 passed: Optimized implementation")

# Test 3: Repeated k-mer within one window
expected = {"AA"}
assert set(FindClumps("AAAAAA", 2, 5, 3)) == expected
assert set(FindClumpsOptimized("AAAAAA", 2, 5, 3)) == expected
print("Test 3 passed: Repeated k-mer within a window")

# Test 4: No clumps
assert set(FindClumps("ACGTACGT", 3, 5, 3)) == set()
assert set(FindClumpsOptimized("ACGTACGT", 3, 5, 3)) == set()
print("Test 4 passed: No clumps")

# Test 5: Compare both implementations
test_cases = [("AAAAAA", 2, 5, 3), ("ATATATAT", 2, 6, 2), ("ACGTACGTACGT", 3, 8, 2), ("AAAAACCCCC", 2, 5, 3)]
for text, test_k, test_L, test_t in test_cases:
    direct = set(FindClumps(text, test_k, test_L, test_t))
    optimized = set(FindClumpsOptimized(text, test_k, test_L, test_t))
    assert direct == optimized, (
        f"Different results for {text}, k={test_k}, L={test_L}, t={test_t}"
    )
print("Test 5 passed: Both implementations agree")

# Test 6: Reject invalid inputs
invalid_inputs = [("ACGT", 0, 3, 2), ("ACGT", 2, 0, 2), ("ACGT", 2, 3, 0), ("ACGT", 4, 3, 1), ("ACGT", True, 3, 2), ("ACGT", 2, 3.5, 1), (None, 2, 3, 1)]
for args in invalid_inputs:
    for function in (FindClumps, FindClumpsOptimized):
        try:
            function(*args)
        except ValueError:
            continue
        raise AssertionError(
            f"{function.__name__} accepted invalid input: {args}"
        )
print("Test 6 passed: Invalid inputs rejected")

# Test 7: Sequence shorter than the window
assert FindClumps("ACGT", 2, 6, 2) == []
assert FindClumpsOptimized("ACGT", 2, 6, 2) == []
print("Test 7 passed: Sequence shorter than window")

# Test 8: Compare implementations across edge cases
test_cases = [("AAAAAA", 2, 5, 3), ("ATATATAT", 2, 6, 2), ("ACGTACGTACGT", 3, 8, 2), ("AAAAACCCCC", 2, 5, 3), ("ACGT", 2, 4, 1), ("ACGT", 2, 6, 2), ("AA", 1, 1, 1)]

for text, test_k, test_L, test_t in test_cases:
    direct = set(FindClumps(text, test_k, test_L, test_t))
    optimized = set(FindClumpsOptimized(text, test_k, test_L, test_t))
    assert direct == optimized, (
        f"Different results for {text}, k={test_k}, L={test_L}, t={test_t}"
    )
print("Test 8 passed: Both implementations agree across edge cases")


Test 1 passed: Basic clump finding
Test 2 passed: Optimized implementation
Test 3 passed: Repeated k-mer within a window
Test 4 passed: No clumps
Test 5 passed: Both implementations agree
Test 6 passed: Invalid inputs rejected
Test 7 passed: Sequence shorter than window
Test 8 passed: Both implementations agree across edge cases
